# TN1 — MixLinear, 63 tham số

## Đây KHÔNG phải phép so cùng ngân sách

Tám cấu hình TN1 khác đều cố ý đặt quanh 56–57 nghìn tham số, để hỏi "cùng
ngân sách thì kiến trúc nào hơn". MixLinear có **63 tham số**, ít hơn LSTM-67
gần **900 lần**. Nó không thuộc phép so đó.

Nó hỏi một câu khác: **ít tham số tới mức nào thì vẫn chạm được mức 0,74?**

Câu hỏi ấy quan trọng vì kết quả TN1 tới giờ như sau:

    tám kiến trúc, từ 55.667 tới 1.502.713 tham số
    điểm cv nằm trong dải     0,7398 - 0,7570      rộng 0,017
    trần trên của chọn kênh          0,9120

Đổi kiến trúc dịch được 0,017, trong khi 0,155 nằm đó không ai lấy. Nếu 63 con
số cũng lên tới quanh 0,74 thì đó là bằng chứng mạnh rằng nút thắt nằm ở **tiêu
chí chọn kênh**, không phải ở sức chứa của mô hình.

Còn nếu nó tụt hẳn thì biết được sức chứa vẫn có vai trò, chỉ là bão hoà sớm.

## Bản cài đặt lấy từ đâu

Từ mã của tác giả: `github.com/aitianma/MixLinear`, `models/MixLinear.py`,
lớp `Model`. Không viết theo mô tả.

## Cách nó hoạt động

    (batch, 200)
      -> trừ trung bình của chính cửa sổ
      -> Conv1d(1, 1, kernel=11) rồi cộng nối tắt        làm mượt
      -> chia thành 20 đoạn, mỗi đoạn 10 mẫu
           |
           +-- nhánh THỜI GIAN: đệm 20 lên 25, xếp thành lưới 5x5,
           |   ép hai chiều của lưới bằng hai lớp 5 -> 2
           |
           +-- nhánh TẦN SỐ: FFT dọc trục 20 đoạn, giữ 5 hệ số tần thấp,
               hai lớp trọng số SỐ PHỨC, rồi FFT ngược
           |
      -> trộn hai nhánh, mỗi bên 0,5, cộng lại trung bình
      -> lấy 25 mẫu đầu

**Vì sao chỉ 63 tham số:** không có lớp nào ánh xạ 200 chiều xuống 25 chiều.
Mọi phép nén đều làm trên trục ĐOẠN (20 hoặc 5 phần tử), còn 10 mẫu trong mỗi
đoạn thì đi song song dùng chung trọng số.

Mẹo chính: thay vì một lớp 20 → 4 tốn 80 trọng số, xếp 20 đoạn thành lưới vuông
rồi ép từng chiều bằng lớp 5 → 2. Hai lớp như vậy tốn 10 + 10 = 20 trọng số mà
vẫn trộn được mọi đoạn với nhau.

`period_len = 10` là cỡ chia đoạn bên trong mạng, **không phải** khẳng định một
nhịp thở dài 10 mẫu. Ở 50 Hz thì 10 mẫu là 0,2 giây, còn một nhịp thở khoảng 4
giây tức 200 mẫu — đúng bằng cả cửa sổ. FFT chạy dọc trục 20 đoạn nên nó hỏi
"biên độ đoạn thay đổi tuần hoàn thế nào qua 4 giây", đúng thang nhịp thở.

## Đếm tham số: 47 hay 63

Hai lớp nhánh tần số có trọng số **số phức**. Một số phức là hai con số thật,
khi train thì cả hai đều được cập nhật — nhưng `numel()` của PyTorch đếm nó là
một.

    TLinear1, TLinear2, conv1d        31 số thật
    FLinear1, FLinear2   16 số phức = 32 số thật
    numel() báo 47, số thật là 63

Đồ án ghi **63**, và `count_params` đã sửa để nhân đôi phần phức. Tám model
khác không có tham số phức nên số của chúng không đổi.

## Ba chỗ cố ý lệch mã tác giả

1. **Bỏ hai lệnh `print("shape", ...)` trong `forward`.** Mã gốc bỏ quên chúng.
   292.708 cửa sổ × 20 epoch × 4 fold thì ngập màn hình và chậm hẳn.

2. **`torch.fft.ifft(...).float()` đổi thành `.real`.** Bản gốc ném cảnh báo
   "Casting complex values to real discards the imaginary part" mỗi lượt gọi.
   Đã đối chiếu bằng `torch.equal`: hai cách cho ra **đúng cùng giá trị**.

3. **Bỏ đối tượng `configs`**, nhận tham số rời, và thêm phần đổi hình dạng
   `(batch, 200)` sang `(batch, 200, 1)` rồi ngược lại ở đầu ra.

**Giữ nguyên** phần trừ rồi cộng lại trung bình của tác giả. Đáng chú ý: nó chỉ
trừ trung bình, **không chia độ lệch chuẩn**, nên biên độ được giữ nguyên. Ở
TN2, RevIN chia độ lệch chuẩn đã kéo DS-TCN xuống 0,0124 vì xoá mất biên độ —
thứ đo được có tương quan 0,53 với chất lượng kênh. MixLinear không dính bẫy đó.

## Rủi ro phải nói trước

63 tham số với lr 1e-4 và 20 epoch **có thể chưa học xong**. Giao thức TN1 dùng
chung siêu tham số của MobiVital cho mọi cấu hình, và đó là điều đúng để so
sánh — nhưng với model bé thế này, điểm thấp có thể là "chưa train đủ" chứ
không phải "kiến trúc không hợp".

Cách phân biệt, không tốn thêm giờ chạy: đọc `curve.csv` sau khi xong. Nếu
`train_mse` vẫn giảm đều ở epoch 19 thì chưa hội tụ; nếu phẳng từ epoch 5 thì
đã học hết những gì học được.

## 1. Chuẩn bị Colab

Mount Drive để lấy cửa sổ train đã cắt ở `DATA_PREPARE.ipynb`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Tải mã nguồn rồi vào thư mục đó. Xem dòng `commit đồ án` để chắc đang chạy bản mới.

In [ ]:
# Xoá trước để chạy lại ô này luôn lấy mã mới nhất, không dính bản cũ.
!rm -rf /content/UWB_RADAR
!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py

Lấy `by_user/` và `windows/` từ Drive. Không cần CSV thô 13 GB.

In [ ]:
!python scripts/restore_processed_data_on_drive.py

## 2. Kiểm bản cài đặt

Mười phép kiểm, vài giây. Sáu phép riêng cho MixLinear nhắm đúng những chỗ đã
nêu ở đầu:

    mục 5    in cả hai cách đếm, xác nhận 47 numel và 63 số thật
    mục 6    tham số phức thật sự nhận gradient, và Adam đổi được chúng —
             không hiển nhiên, vì tối ưu trên số phức là đường riêng của torch
    mục 7    quét mã nguồn forward, xác nhận không còn lệnh print nào
    mục 8    xoá trọng số từng nhánh, đầu ra phải đổi — chứng tỏ cả nhánh
             thời gian lẫn nhánh tần số đều thật sự nối vào
    mục 9    dựng lại phép trộn bằng tay, xác nhận mix_alpha đúng là trọng số
    mục 10   chia đúng 20 đoạn và lưới 5x5

In [ ]:
!python scripts/check_model.py --model mix_linear --period_len 10 --lpf 5

## 3. MixLinear — 4 fold CV, 3 seed

Mười phép kiểm đạt thì mới chạy.

Tên cấu hình là `mix_linear_p10_lpf5_mse_corr0.9_seed<N>`. Hai con số `p10` và
`lpf5` LUÔN được ghi vì chúng quyết định cả kiến trúc lẫn số tham số.

Train gần như tức thì vì model quá bé, nhưng phần chấm điểm vẫn khoảng 25 phút
mỗi seed và không đổi theo kiến trúc. Ước lượng **khoảng 1,5 giờ**.

In [ ]:
!python scripts/run_cv.py --experiment tn1 --model mix_linear --seed 0
!python scripts/run_cv.py --experiment tn1 --model mix_linear --seed 1
!python scripts/run_cv.py --experiment tn1 --model mix_linear --seed 2

## 4. Cất kết quả

Nén lại một lần sau khi xong cả 12 lần chạy, ra tên riêng `tn1_mix_linear.zip`.

In [ ]:
!python scripts/save_results.py tn1 --out tn1_mix_linear

## 5. Đường hội tụ

Ô này trả lời câu hỏi ở phần rủi ro: 20 epoch có đủ cho một model 63 tham số
không. In `train_mse` của fold đầu, seed 0.

Còn giảm đều ở epoch 19 nghĩa là chưa hội tụ, và điểm thấp không kết luận được
gì về kiến trúc. Phẳng từ giữa nghĩa là đã học hết.

In [ ]:
!head -1 runs/tn1/mix_linear_p10_lpf5_mse_corr0.9_seed0_val_AB/curve.csv
!cut -d, -f1,2 runs/tn1/mix_linear_p10_lpf5_mse_corr0.9_seed0_val_AB/curve.csv | tail -21

## 6. Bảng so trong phiên này

In [ ]:
!python scripts/compare_cv.py --experiment tn1

## 7. Ngắt phiên

Colab giữ runtime sau khi ô cuối chạy xong và vẫn tính giờ. Ô này đóng phiên
lại. Kết quả đã nén sang Drive nên ngắt ở đây không mất gì.

Bấm liên tiếp các ô trên thì Colab xếp hàng chạy lần lượt, không cần ngồi canh.

In [ ]:
from google.colab import runtime
runtime.unassign()